# Geração de patches e manifesto de dados por fonte (estágio 06)

Divide, para cada fonte de ground truth habilitada, o composite multiespectral normalizado do ano da fonte (estágio 05) e a máscara binária da própria fonte (estágio 02) em patches de `data.patch_size` (512x512 px, sem sobreposição), filtrando pela proporção mínima de café. Cada fonte tem seu próprio conjunto de patches e manifesto (`data/processed/manifest_<fonte>.parquet`), com a divisão de treino/validação feita no estágio 07.

## Bootstrap do workspace

O primeiro passo baixa e executa `src/bootstrap.py` (somente stdlib) — necessário porque o `src/` ainda não está disponível para import em uma sessão nova. O bootstrap obtém o repositório público, extrai `src/`, `data/external/` e `requirements-runtime.txt` para o workspace e adiciona o workspace ao `sys.path`. O `reload` garante que reexecuções usem a versão mais recente baixada.

In [ ]:
# Baixa e executa o bootstrap do workspace (etapa prévia ao import de src/).
import importlib
import pathlib
import sys
import urllib.request

BOOTSTRAP_URL = "https://raw.githubusercontent.com/jotap1101/tcc/main/src/bootstrap.py"
pathlib.Path("bootstrap.py").write_bytes(urllib.request.urlopen(BOOTSTRAP_URL).read())
sys.path.insert(0, str(pathlib.Path.cwd()))

# Recarrega o módulo para não reutilizar uma versão antiga em cache no kernel.
bootstrap = importlib.import_module("bootstrap")
importlib.reload(bootstrap)

workspace = bootstrap.bootstrap_workspace()
print(f"Workspace: {workspace}")

## Dependências pinadas

Instala as versões fixadas em `requirements-runtime.txt` (incluindo rasterio, usado na leitura dos GeoTIFFs, e pandas/pyarrow, usados na escrita do manifesto), garantindo o mesmo conjunto de bibliotecas nas duas plataformas.

In [ ]:
# Instala as versões pinadas do requirements-runtime.txt no ambiente atual.
import subprocess
import sys

requirements = pathlib.Path(workspace) / "requirements-runtime.txt"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(requirements)],
    check=True,
)
print("Dependências instaladas a partir de:", requirements)

## Pacote compartilhado, plataforma e armazenamento

Importa o pacote `src/` (já entregue pelo bootstrap) e identifica a plataforma pela abstração em `src/io.py`. Em seguida garante a raiz `MyDrive/tcc/` e resolve todos os caminhos de armazenamento definidos em `src/config.yaml`, incluindo a subpasta de patches e o manifesto.

In [ ]:
# Resolve o contexto de runtime: plataforma, armazenamento canônico e configuração.
from src.setup import initialize

ctx = initialize(workspace)
platform = ctx.platform
storage_paths = ctx.storage_paths
config = ctx.config
print(f"Plataforma: {platform}")
print(f"Patches por fonte: {storage_paths['data_processed_patches']}/<fonte>")
print(f"Figuras: {storage_paths['artifacts_figures']}")


## Reprodutibilidade

Fixa as sementes de python/numpy/torch/cuda e habilita as flags determinísticas do PyTorch, além de registrar as versões dos pacotes principais do ambiente, garantindo o mesmo protocolo de execução nas duas plataformas.

In [ ]:
# Fixa sementes, flags determinísticas e registra as versões do ambiente.
from src.utils import log_environment, setup_reproducibility

setup_reproducibility(config)
log_environment()

## Dependências dos estágios 05 e 02

Verifica, para cada fonte habilitada, que o composite normalizado do ano da fonte (estágio 05) e a máscara binária da própria fonte (estágio 02) estão disponíveis no caminho canônico — sem estas dependências a geração de patches da fonte não pode ocorrer.

In [ ]:
# Verifica as dependências do estágio 05 (composite do ano) e do estágio 02 (máscara da fonte).
from src.data.mask_utils import reference_year, source_mask_path
from src.data.preprocessing import composite_path
from src.utils import check_dependencies

enabled_sources = [
    name for name, cfg in config["ground_truth"]["sources"].items() if cfg["enabled"]
]
dependencies = {}
for name in enabled_sources:
    dependencies[f"Composite {reference_year(name)} (estágio 05)"] = composite_path(
        storage_paths, reference_year(name)
    )
    dependencies[f"Máscara {name} (estágio 02)"] = source_mask_path(name, storage_paths)
check_dependencies(dependencies)


## Geração de patches por fonte

Garante (idempotente) os patches 512x512 e o manifesto de cada fonte em `MyDrive/tcc/data/processed/`: o composite do ano da fonte é cortado no grid da máscara da própria fonte (mesmo grid, sem sobreposição), e cada patch é registrado no manifesto com fingerprint de versão.

In [ ]:
# Garante os patches e o manifesto de cada fonte (reutiliza o que já existe).
from src.data.patch_generation import generate_patches

manifest_files = {name: generate_patches(storage_paths, name) for name in enabled_sources}


## Verificação dos manifestos

Confere o manifesto persistido de cada fonte: total de patches, distribuição da proporção de café, formato dos arrays e fonte de máscara registrada.

In [ ]:
# Verifica o manifesto persistido de cada fonte (contagem, café e formato).
from src.data.patch_generation import verify_manifest

manifest_stats = {}
for name in enabled_sources:
    manifest_stats[name] = verify_manifest(storage_paths, name)
    stats = manifest_stats[name]
    coffee = stats["coffee_ratio"]
    print(
        f"{name}: {stats['n_patches']} patches (com café: {stats['n_with_coffee']}) | "
        f"café [mín {coffee['min']:.3f}, média {coffee['mean']:.3f}, máx {coffee['max']:.3f}] | "
        f"forma {stats['patch_shape']}"
    )


## Mosaicos de amostras por fonte

Renderiza e persiste, para cada fonte, um mosaico com as amostras mais cafeeiras (RGB do composite + máscara binária) em `MyDrive/tcc/artifacts/figures/`; execuções repetidas reutilizam as figuras existentes.

In [ ]:
# Renderiza e persiste a figura do mosaico de amostras de cada fonte (reutiliza se já existir).
from src.data.patch_generation import save_patch_montage

montage_paths = {name: save_patch_montage(storage_paths, name) for name in enabled_sources}


## Resumo da etapa

Exibe o resumo da geração de patches por fonte: fontes processadas, manifestos persistidos, estatísticas de verificação e figuras de registro.

In [ ]:
# Exibe o resumo da etapa de geração de patches por fonte.
summary = {
    "Fontes processadas": ", ".join(enabled_sources),
    "Manifestos": ", ".join(str(path) for path in manifest_files.values()),
    "Patches registrados": {name: stats["n_patches"] for name, stats in manifest_stats.items()},
    "Formato dos patches": next(iter(manifest_stats.values()))["patch_shape"],
    "Figuras de amostras": ", ".join(str(path) for path in montage_paths.values()),
}
from src.utils import print_summary

print_summary(summary, "06")
